# 01 LIBERO 环境与数据学习

> **目标**：理解 LIBERO 基准的「环境 + 数据 + 语言指令」，把 PushT 学到的闭环推理框架迁移到 3D 机械臂操作。
> 环境：**Franka 机械臂 + MuJoCo**（与 PushT 同一引擎）；框架：**LeRobot 0.6.1 内置支持**。

## 1. LIBERO 是什么

**LIBERO 不是模型，是「任务基准 + 数据集」**（LifeLong Robot Learning Benchmark）：
- **130 个桌面操作任务**，4 个套件：Spatial（空间泛化）/ Object（物体泛化）/ Goal（目标泛化）/ Long（长时程）
- 每个任务有**语言指令**（如 "pick up the black bowl and place it on the plate"）——这是 VLA 的关键
- 观测 = 2 视角图像（agentview + 腕部相机）；动作 = 7-DoF 关节相对增量（range [-1,1]）


In [ ]:
# 0. 初始化 LIBERO 路径（必须最先执行，在 import libero 之前）
import os, sys
os.environ["LIBERO_CONFIG_PATH"] = os.path.join(os.getcwd(), ".libero")
from pathlib import Path
cfg = Path(os.environ["LIBERO_CONFIG_PATH"])
cfg.mkdir(parents=True, exist_ok=True)
if not (cfg / "config.yaml").exists():
    import yaml
    sitepkg = Path(sys.prefix) / "Lib" / "site-packages" / "libero" / "libero"
    yaml.safe_dump({
        "benchmark_root": str(sitepkg),
        "bddl_files": str(sitepkg / "bddl_files"),
        "init_states": str(sitepkg / "init_files"),
        "datasets": str(Path.cwd().parent.parent / "datasets"),
    }, open(cfg / "config.yaml", "w", encoding="utf-8"))
print("LIBERO 配置就绪:", cfg / "config.yaml")

## 2. 环境：观测 / 动作 / 任务

`lerobot.envs.libero.LiberoEnv` 关键点：
- 观测 `pixels`: `image`（agentview 俯视） + `image2`（腕部相机），256×256 RGB
- （可选 `pixels_agent_pos` 加 `robot_state`: eef/gripper/joints）
- 动作：`Box(-1, 1, (7,))` —— **相对控制**（关节增量）
- 每个任务带 `task_description`（语言指令）与对应的 bddl 任务文件


In [ ]:
# 1. 创建环境并检查空间
import gymnasium as gym
from lerobot.envs.libero import create_libero_envs, get_libero_dummy_action

envs = create_libero_envs("libero_spatial", n_envs=1,
                          env_cls=gym.vector.SyncVectorEnv,
                          gym_kwargs={"task_ids": [0]})
env = envs["libero_spatial"][0]
print("action_space:", env.action_space)
print("obs space keys:", {k: (v.shape if hasattr(v, "shape") else list(v.keys())) for k, v in env.observation_space.spaces.items()})
print("dummy action:", get_libero_dummy_action(env))

In [ ]:
# 2. reset + 一步，观察真实观测与渲染
obs, info = env.reset(seed=0)
print("obs keys:", sorted(obs.keys()))
for k, v in obs["pixels"].items():
    print(f"  pixels[{k}]:", v.shape, v.dtype)
img = obs["pixels"]["image"][0]  # (H,W,3)
print("agentview 图像 min/max/mean:", img.min(), img.max(), img.mean().round(2))
# 语言指令（任务描述）——每个 task 自带
print("task 描述:", envs["libero_spatial"][0].task_description)

## 3. 数据集：`lerobot/libero`（~2GB，公开）

- 包含 4 个套件的**全部演示**（Spatial/Object/Goal/Long），按 `suite` 字段区分
- 特征：`observation.images.image / image2`（256×256）、`observation.state`、`action`、**`language_instruction`**、`episode_index` 等
- 训练时用 `--dataset.repo_id=lerobot/libero --env.task=libero_spatial` 自动按套件筛选


In [ ]:
# 3. 加载数据集并查看结构（首次运行会从 HF 下载 ~2GB 到 datasets/hub）
import os
os.environ.setdefault("HF_HOME", os.path.abspath("../datasets"))
from datasets import load_dataset
ds = load_dataset("lerobot/libero", split="train")
print("总行数:", len(ds), "| 套件:", sorted(set(ds["suite"])))
print("特征:", [k for k in ds.features.keys()])
# 一个 episode 的样本数
import numpy as np
ep = ds[0]
print("episode_index:", ep["episode_index"], "| suite:", ep["suite"])
print("language_instruction:", ep["language_instruction"])
print("image:", ep["observation.images.image"].shape, "| action:", ep["action"].shape, "| state:", ep["observation.state"].shape)
# 每个套件有多少 episode
from collections import Counter
eps = {}
for s in sorted(set(ds["suite"])):
    eps[s] = len(set(ds["episode_index"][ds["suite"] == s]))
print("各套件 episode 数:", eps)

## 4. 闭环推理骨架（与 PushT 完全同构）

```
obs(图像+状态) → preprocessor → policy.select_action → postprocessor → env.step
```
区别只是：观测多了「语言指令」，动作从 2D 位置变成 7-DoF 关节增量。
下面用 **随机策略** 先跑通闭环；换上 ACT/SmolVLA 权重即真实推理。


In [ ]:
# 4. 随机策略闭环 5 步（验证 obs->action->step 通路）
import numpy as np
obs, info = env.reset(seed=0)
total = 0.0
for i in range(5):
    act = env.action_space.sample()
    obs, rew, term, trunc, info = env.step(act)
    total += float(rew.sum())
print("5 步随机闭环完成, 累计 reward:", round(float(total), 4))
env.close()

## 5. ACT 训练（模仿学习）

```bash
python -m lerobot.scripts.lerobot_train \
  --env.type=libero --env.task=libero_spatial --env.task_ids=[0] \
  --dataset.repo_id=lerobot/libero --dataset.sampling_ratio=1.0 \
  --policy.type=act --policy.chunk_size=100 \
  --output_dir=outputs/train/libero_act --steps=50000 --batch_size=8 \
  --save_freq=5000 --wandb.enable=false
```

- `--env.type=libero`：用 LeRobot 内置 LIBERO 环境做闭环评估
- `--env.task=libero_spatial`：指定套件（训练时数据也按此筛选）
- 8GB 显卡建议：`batch_size=8`、图像 256×256（ACT 视觉骨干）
- 评估：`python -m lerobot.scripts.lerobot_eval --env.type=libero --env.task=libero_spatial --policy.path=<checkpoint>`

## 6. 进阶：VLA（SmolVLA）

- 在**同一数据集**上换 `--policy.type=smolvla` 即训练 VLA（视觉+语言→动作）
- 语言指令就是 `language_instruction` 字段 —— 数据无需重新采集
- 参考：https://github.com/huggingface/lerobot/blob/main/docs/source/smolvla.mdx


## 7. 用预训练权重跑推理（2026-08-17 实测）

**完整闭环已在 Windows 验证**（环境→数据→训练→评估）。社区/官方预训练权重：

| 权重 | 说明 | 实测结果 |
|---|---|---|
| `ishandotsh/act_libero_spatial_test` | 社区 ACT（spatial） | 10 局跑通，0/10 成功（机器人真实运动，checkpoint 本身弱） |
| `Deepkar/libero-test-act` | 社区 ACT（libero_10，100k 步） | spatial 0/1（跨套件）；libero_10 评估挂起（待排查） |
| `HuggingFaceVLA/smolvla_libero` | **官方 SmolVLA**（VLA） | **LIBERO-Spatial task0 成功率 80%（4/5）** ✅ |

**Windows 评估要点**（`lerobot-eval`）：
```bash
python -m lerobot.scripts.lerobot_eval --env.type=libero --env.task=libero_spatial \
    --env.task_ids=[0] --env.max_parallel_tasks=1 --eval.use_async_envs=false \
    --policy.path=<权重目录> --eval.n_episodes=10
```
- `--env.max_parallel_tasks=1` + `--eval.use_async_envs=false`：**Windows 必须**（AsyncVectorEnv 会挂起）
- 评估输出视频/指标到 `outputs/eval/<时间戳>_<模型>/`

**SmolVLA 下载/评估要点**：`HuggingFaceVLA/smolvla_libero` 依赖基础 VLM `HuggingFaceTB/SmolVLM2-500M-Instruct`
（~2GB；仓库共 7.9GB 含 5GB ONNX 导出，**只需 `model.safetensors` + 配置文件**）。国内网络受限时用 hf-mirror：
```bash
curl -L --proxy http://127.0.0.1:7897 -C - -o model.safetensors \
  https://huggingface.co/HuggingFaceTB/SmolVLM2-500M-Instruct/resolve/main/model.safetensors
```
